In [71]:
# create a tree of categories of FGL sources

In [1]:
import numpy as np
from astropy.io import fits
from matplotlib import pyplot
import pandas as pd
import os
import copy

from sklearn.mixture import GaussianMixture

import sys
if '../lib/' not in sys.path:
    sys.path.append('../lib/')

import dio3
import multi_fgl_utils as util


In [2]:
if 1:
    import importlib
    importlib.reload(util);
np.set_printoptions(precision=4)

In [9]:
save_res = 0
cat_name = '4FGL-DR4' # '4FGL-DR3'
case = None
if cat_name == '4FGL-DR4':
    baseline_case = '4FGL-DR4_GMM_RF'
    #case = '4FGL-DR4_GMM_RF'
    #case = '4FGL-DR4_GMM_RF_no_bcu_spp'
    #case = '4FGL-DR4_GMM_RF_no_bcu'
    #case = '4FGL-DR4_GMM_RF_no_coord'
    case = '4FGL-DR4_GMM_RF_no_coord_no_bcu_spp'
    #case = '4FGL-DR4_GMM_RF_no_coord_no_bcu'
    #baseline_case = '4FGL-DR4_GMM_RF_no_coord_no_bcu_spp'
    #case = '4FGL-DR4_GMM_RF_no_coord_no_bcu_spp' # '4FGL-DR4_GMM_RF_no_prune'
    #case = '4FGL-DR4_GMM_RF_no_coord_no_bcu_spp_nmin50'
    #baseline_case = '4FGL-DR4_GMM_RF_no_coord'
    #case = '4FGL-DR4_GMM_RF_no_coord'
if cat_name == '4FGL-DR3':
    baseline_case = '4FGL-DR3_GMM_RF_no_coord_no_bcu_spp'
    #case = '4FGL-DR3_GMM_RF_no_coord_no_bcu_spp' # '4FGL-DR4_GMM_RF_no_prune'
    case = '4FGL-DR3_GMM_RF_no_coord_no_bcu_spp_nmin50'

if case is None:
    case = baseline_case
    
baseline = (case == baseline_case)
root_folder = '../'
config_folder = root_folder + 'config/'

# generic inputs
cdict = util.get_config_dict(#cat_name=cat_name, 
                             baseline_case=baseline_case, 
                             #baseline_fn=baseline_fn,
                             #cases_fn=cases_fn,
                             case=case, 
                             folder=config_folder)

folder_in = cdict.get('folder_in', root_folder + 'data_in/')
folder_out = cdict.get('folder_out', root_folder + 'data_out/')
Eind = int(cdict.get('Eind', 1000))
add_plec_pars = int(cdict.get('add_plec_pars', 0))

assoc_sources = cdict.get('assoc_sources', 'all_assoc')
sep_model = cdict.get('sep_model', 'GMM')
#sep_model = 'RF' # 'GMM', 'RF'

nmin = int(cdict.get('nmin', 100))
nmin_GMM = cdict.get('nmin_GMM')
prune = cdict.get('prune')
if prune == 'None':
    prune = None
    cdict['prune'] = None
if prune is not None:
    prune = int(prune)
max_depth = int(cdict.get('max_depth', 5))


#folder_in = '../data_in/'
#folder_out = '../data_out/'

#out_fn = folder_out + 'cat_tree_nmin%i_%s_%s_%s.yaml' % (nmin, sep_model, cat_name, assoc_sources)
tree_cat_fn = util.get_tree_cat_fn(cdict)
cdict['prune'] = 'None'
tree_cat_no_prune_fn = util.get_tree_cat_fn(cdict)
cdict['prune'] = prune
#tree_cat_fn = folder_out + 'tree_cat_nmin%i%s%s.yaml' % (nmin, sep_str, mode_str)



xdf_fn = util.get_xdf_fn(cdict)
xdf = pd.read_csv(xdf_fn, index_col=0)

class_name = cdict.get('class_name', 'CLASS1')
feature_mode = cdict.get('features', 'all_features')
all_features = [name for name in xdf.columns if name.find('CLASS') == -1]
if feature_mode == 'all_features':
    features = all_features
elif feature_mode == 'no_coord_features':
    features = [feat for feat in all_features if feat.find('GL') == -1]

ns = len(xdf)

pars_dict = util.pars_dict
pars_dict['NN']['nfeatures'] = len(features)
pars_dict['NN']['epochs'] = cdict.get('epochs', 500)


nsrc_dict = util.get_nsrc_dict(xdf, class_column=class_name, classes='associated')
if assoc_sources == 'no_bcu_spp':
    excl_cls = ['bcu', 'spp', 'unas', 'unk']
    assoc_classes = [cl for cl in list(set(xdf[class_name])) if cl not in excl_cls]
elif assoc_sources == 'no_bcu':
    excl_cls = ['bcu', 'unas', 'unk']
    assoc_classes = [cl for cl in list(set(xdf[class_name])) if cl not in excl_cls]
else:
    assoc_classes = list(nsrc_dict.keys())


In [42]:
xdf_fn

'../data_out/xdf_4FGL-DR4_Eind1000MeV.csv'

In [10]:
prune is None, prune

(True, None)

In [11]:
cdict

{'cat_name': '4FGL-DR4',
 'version': 34,
 'Eind': 1000,
 'add_plec_pars': 0,
 'sep_model': 'GMM',
 'nmin': 100,
 'nmin_GMM': 10,
 'max_depth': 5,
 'assoc_sources': 'no_bcu_spp',
 'unas_sources': 'unas_unk',
 'features': 'no_coord_features',
 'class_alg': 'RF',
 'max_sample': 10,
 'nGMM_w': 12,
 'test_frac': 0.3,
 'nav': 10,
 'epochs': 1000,
 'root_folder': '../',
 'folder_out': 'data_out/',
 'folder_in': 'data_in/',
 'case': '4FGL-DR4_GMM_RF_no_coord_no_bcu_spp',
 'baseline_case': '4FGL-DR4_GMM_RF',
 'prune': None}

In [12]:
features

['log10(Energy_Flux100)',
 'log10(Unc_Energy_Flux100)',
 'log10(Signif_Avg)',
 'LP_index1000MeV',
 'LP_beta',
 'LP_SigCurv',
 'log10(Variability_Index)']

In [13]:
tree_cat_fn, tree_cat_no_prune_fn

('../data_out//tree_cat_4FGL-DR4_sep_model_GMM_nmin_100_no_bcu_spp_no_coord_features.yaml',
 '../data_out//tree_cat_4FGL-DR4_sep_model_GMM_nmin_100_prune_None_no_bcu_spp_no_coord_features.yaml')

In [14]:
%%time
cat_dict = {cl:"0" for cl in assoc_classes}
#cat_dict = {cl:[0] for cl in assoc_classes}
cls_in = assoc_classes
util.get_tree_cats(cat_dict, cls_in, xdf, cname=class_name, features=features, nmin=nmin, 
                   max_depth=max_depth, nsrc_dict=nsrc_dict, sep_model=sep_model, nmin_GMM=nmin_GMM)
# remove common root ?
#for cl, lst in cat_dict.items():
#    cat_dict[cl] = lst[1:]

['psr', 'pwn', 'snr', 'fsrq', 'rdg', 'bin', 'msp', 'glc', 'bll', 'hmb']
['fsrq', 'rdg', 'bll']
['fsrq']
['rdg', 'bll']
['psr', 'pwn', 'snr', 'bin', 'msp', 'glc', 'hmb']
['psr', 'pwn', 'snr', 'hmb']
['bin', 'msp', 'glc']
CPU times: user 1.23 s, sys: 1.04 s, total: 2.27 s
Wall time: 364 ms


In [15]:
s = ''
for cl in np.sort(list(cat_dict.keys())):
    s += '%s, ' %cl
print(s[:-2])

agn, bin, bll, css, fsrq, gal, gc, glc, hmb, lmb, msp, nlsy1, nov, psr, pwn, rdg, sbg, sey, sfr, snr, ssrq


In [16]:
ncl = 0
nout = 0
ncat = 0
print(len(cat_dict), len(assoc_classes))
while nout < len(cat_dict):
    printed = 0
    for cl, leaf in cat_dict.items():
        #charv = np.sum(2**(np.arange(len(cats))) * np.array(cats))
        charv = int(leaf[::-1], base=2)
        if charv == ncl:
            print(cl, leaf, nsrc_dict[cl])
            nout += 1
            printed = 1
    if printed:
        ncat += 1
        print()
    ncl += 1
print(ncat)

21 21
nlsy1 000 8
css 000 6
fsrq 000 819

psr 010 141
pwn 010 21
snr 010 45
nov 010 6
gc 010 1
hmb 010 11

ssrq 001 2
sbg 001 8
agn 001 8
rdg 001 53
sey 001 4
bll 001 1490

gal 011 6
lmb 011 9
sfr 011 6
bin 011 10
msp 011 179
glc 011 34

4


In [17]:
tree_cat = util.cat_dict2tree_cat(cat_dict)
print(len(tree_cat))
for node, cls in tree_cat.items():
    nsrc = np.sum([nsrc_dict[cl] for cl in cls])
    print(node, cls, nsrc)

4
001 ['ssrq', 'sbg', 'agn', 'rdg', 'sey', 'bll'] 1565
011 ['gal', 'lmb', 'sfr', 'bin', 'msp', 'glc'] 244
000 ['nlsy1', 'css', 'fsrq'] 833
010 ['psr', 'pwn', 'snr', 'nov', 'gc', 'hmb'] 225


In [18]:
tree_cat0 = copy.deepcopy(tree_cat)

In [19]:
if prune is not None:
    min_nsrc = np.min([np.sum([nsrc_dict[cl] for cl in cls]) for cls in tree_cat.values()])
    while min_nsrc < prune:
        print(min_nsrc)
        tree_cat = util.prune_tree_cat(tree_cat, xdf, prune_alg='RF', max_depth=max_depth, features=features,
                                   pars_dict=pars_dict, nsrc_dict=nsrc_dict, class_name=class_name)
        
        min_nsrc = np.min([np.sum([nsrc_dict[cl] for cl in cls]) for cls in tree_cat.values()])

In [20]:
for node, cls in tree_cat.items():
    nsrc = np.sum([nsrc_dict[cl] for cl in cls])
    print(node, cls, nsrc)

001 ['ssrq', 'sbg', 'agn', 'rdg', 'sey', 'bll'] 1565
011 ['gal', 'lmb', 'sfr', 'bin', 'msp', 'glc'] 244
000 ['nlsy1', 'css', 'fsrq'] 833
010 ['psr', 'pwn', 'snr', 'nov', 'gc', 'hmb'] 225


In [21]:
cat_dict

{'ssrq': '001',
 'gal': '011',
 'sbg': '001',
 'lmb': '011',
 'nlsy1': '000',
 'sfr': '011',
 'psr': '010',
 'css': '000',
 'pwn': '010',
 'snr': '010',
 'nov': '010',
 'agn': '001',
 'fsrq': '000',
 'rdg': '001',
 'bin': '011',
 'msp': '011',
 'glc': '011',
 'sey': '001',
 'gc': '010',
 'bll': '001',
 'hmb': '010'}

In [22]:
#dio3.savedict(cat_dict, out_fn)
if save_res:
    if prune is not None:
        dio3.savedict(tree_cat0, tree_cat_no_prune_fn)
    dio3.savedict(tree_cat, tree_cat_fn)

In [23]:
cat_tree = util.cat_dict2tree(cat_dict)
cat_tree

['000', '001', '010', '011']

In [24]:
#[str2branch(st) for st in cat_dict2tree_cat(cat_dict_str).keys()]

In [25]:
#branch = [0]
node = "0"
tree = [node]
forest = [tree]


In [26]:
util.is_internal(tree[0], tree=cat_tree)

True

In [27]:
#get_max_branches(tree, max_tree=cat_tree)
util.get_max_nodes(tree, max_tree=cat_tree)

[0]

In [28]:
new_forest0 = util.grow_trees(forest)
new_forest = util.grow_trees(forest, max_tree=cat_tree)

In [29]:
new_forest

[['0'], ['00', '01']]

In [30]:
new_forest = util.grow_trees(new_forest, max_tree=cat_tree)

In [31]:
util.print_forest(new_forest)

['']
['0', '1']
['0', '10', '11']
['1', '00', '01']
['00', '01', '10', '11']


In [32]:
new_forest = util.grow_trees(new_forest, max_tree=cat_tree)

In [33]:
len(new_forest)

5

In [34]:
util.print_forest(new_forest)

['']
['0', '1']
['0', '10', '11']
['1', '00', '01']
['00', '01', '10', '11']


In [35]:
cat_forest = util.get_forest(max_depth=None, max_tree=cat_tree)

In [36]:
len(cat_forest)

5

In [37]:
util.print_forest(cat_forest[-10:])

['']
['0', '1']
['0', '10', '11']
['1', '00', '01']
['00', '01', '10', '11']


In [38]:
util.print_forest([cat_tree])

['00', '01', '10', '11']


In [39]:
# check if the last tree in the forest is the max tree
np.sort(cat_forest[-1]) == np.sort(cat_tree)

array([ True,  True,  True,  True])

In [40]:
features

['log10(Energy_Flux100)',
 'log10(Unc_Energy_Flux100)',
 'log10(Signif_Avg)',
 'LP_index1000MeV',
 'LP_beta',
 'LP_SigCurv',
 'log10(Variability_Index)']

In [41]:
nsrc_dict

{'ssrq': 2,
 'gal': 6,
 'sbg': 8,
 'lmb': 9,
 'nlsy1': 8,
 'sfr': 6,
 'psr': 141,
 'css': 6,
 'pwn': 21,
 'snr': 45,
 'nov': 6,
 'agn': 8,
 'fsrq': 819,
 'rdg': 53,
 'bin': 10,
 'msp': 179,
 'glc': 34,
 'sey': 4,
 'bcu': 1625,
 'gc': 1,
 'bll': 1490,
 'hmb': 11,
 'spp': 124}

In [47]:
sum(xdf['CLASS1'] == 'unas')

2428